In [ ]:
pip install shap

In [ ]:
pip install openpyxl


In [ ]:
import shap
from google.colab import files
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. We will use this in our plotter function to plot data.
import matplotlib.pyplot as plt
#Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns
# Preprocessing allows us to standarsize our data
from sklearn import preprocessing
# Allows us to split our data into training and testing data
from sklearn.model_selection import train_test_split
# Allows us to test parameters of classification algorithms and find the best one
from sklearn.model_selection import GridSearchCV
# Logistic Regression classification algorithm
from sklearn.linear_model import LogisticRegression
# Support Vector Machine classification algorithm
from sklearn.svm import SVC
# Decision Tree classification algorithm
from sklearn.tree import DecisionTreeClassifier
# K Nearest Neighbors classification algorithm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
# Upload your CSV from local
uploaded = files.upload()

Saving question.xlsx to question.xlsx
Saving ResultsForSurvey-552 - 07-07-2026 01_51_11.xlsx to ResultsForSurvey-552 - 07-07-2026 01_51_11.xlsx


In [ ]:
# Load it into a DataFrame
df = pd.read_excel("ResultsForSurvey-552 - 07-07-2026 01_51_11.xlsx")  # replace with actual filename

In [ ]:
# Load it into a DataFrame
df_pitanja = pd.read_excel("question.xlsx")  # replace with actual filename

In [ ]:
df = df.drop(columns=[
    'ResultId', 'Project', 'Survey', 'Language', 'UserGroup',
    'Code', 'AccessId', 'Access', 'Segments', 'CurrentPage','Progress', 'StartTime', 'EndTime','Test'
])


In [ ]:
ordinal_vars = []

for qid, qtype, options in zip(df_pitanja["QuestionID"], df_pitanja["Type"], df_pitanja["Options"]):
    if qtype == "Scale" and pd.notnull(options):
        choices = [opt.strip() for opt in str(options).split(';') if '=' in opt]
        if len(choices) > 2:   # more than 3 options
            ordinal_vars.append(str(qid))

# same cleanup you used (remove .0)
ordinal_vars = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in ordinal_vars]

In [ ]:
select_one = []
for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "SelectOne":
        select_one.append(qid)
select_one = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in select_one]

In [ ]:
multi_nominal_cols = []
for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "SelectMultiple":
        multi_nominal_cols.append(qid)
multi_nominal_cols = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in multi_nominal_cols]

In [ ]:
binary = []

for qid, qtype, options in zip(df_pitanja["QuestionID"], df_pitanja["Type"], df_pitanja["Options"]):
    if qtype == "Scale" and pd.notnull(options):
        choices = [opt.strip() for opt in str(options).split(';') if '=' in opt]
        if len(choices) <= 2:   # <= 3 options
            binary.append(str(qid))

binary = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in binary]

In [ ]:
# See missing values per column
df_mising=df.isnull().sum().sort_values(ascending=False)

In [ ]:
df_mising

,0
uniqaCode,142
2997,140
2045.2,139
2046.1,136
2996,133
...,...
2985,0
593,0
592,0
250.2,0


In [ ]:
za_drop = []

for index, value in df_mising.items():
    if value > 40:
        za_drop.append(index)

In [ ]:
za_drop = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in za_drop]

In [ ]:
za_drop2 = []

for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "Text":
        za_drop2.append(str(qid))  # convert to string here


In [ ]:
za_drop2_cleaned = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in za_drop2]

In [ ]:
# 🔹 Step 2: Drop rows where 379.1 is missing
df = df[df['2001'].notna()]

In [ ]:
df = df.drop(columns=za_drop)

In [ ]:
df = df.drop(columns=za_drop2_cleaned, errors='ignore')


In [ ]:
# Step 2: Filter out columns that don’t exist in the DataFrame
ordinal_vars = [col for col in ordinal_vars if col in df.columns]

# Step 3: Convert only valid ones
for col in ordinal_vars:
    df[col] = df[col].astype('Int64')


In [ ]:
for col in multi_nominal_cols:
    if col in df.columns:
        # Step 1: Clean missing and convert to string
        df[col] = df[col].fillna('').astype(str)

        # Step 2: One-hot encode selections
        dummies = df[col].str.get_dummies(sep=';')
        dummies.columns = [f"{col}_{c.strip()}" for c in dummies.columns]

        # Step 3: Create missing-all flag BEFORE dropping or joining
        df[f'{col}_missing_all'] = dummies.sum(axis=1).eq(0).astype(int)

        # Step 4: Replace original with dummies
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)


In [ ]:
for col in select_one:
    if col in df.columns:
        # Step 1: Clean missing and convert to string
        df[col] = df[col].fillna('').astype(str)

        # Step 2: One-hot encode selections
        dummies = df[col].str.get_dummies(sep=',')
        dummies.columns = [f"{col}_{c.strip()}" for c in dummies.columns]

        # Step 3: Create missing-all flag BEFORE dropping or joining
        df[f'{col}_missing_all'] = dummies.sum(axis=1).eq(0).astype(int)

        # Step 4: Replace original with dummies
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)


In [ ]:
for col in binary:
    if col in df.columns:
        df[col] = df[col].replace({1: 1, 2: 0})

In [ ]:
import numpy as np

# Compute correlation matrix
corr = df.corr()

# Extract upper triangle (to avoid duplicates)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Unstack into long form
strong_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={0: "Correlation", "level_0": "Var1", "level_1": "Var2"})
)

# Filter for absolute correlation above threshold
threshold = 0.8
strong_corr = strong_corr[strong_corr["Correlation"].abs() > threshold]

print(strong_corr)

Empty DataFrame
Columns: [Var1, Var2, Correlation]
Index: []


In [ ]:
cols_to_drop = ['416_2.0']
df = df.drop(columns=cols_to_drop)


In [ ]:
import re

# --- 1. grab all missing flags ---
missing_cols = [c for c in df.columns if c.endswith("_missing_all")]

# --- 2. group them by shared missingness pattern (union-find over the corr graph) ---
threshold = 0.8
mcorr = df[missing_cols].corr().abs()

parent = {c: c for c in missing_cols}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

for i, a in enumerate(missing_cols):
    for b in missing_cols[i + 1:]:
        if mcorr.loc[a, b] >= threshold:   # correlated (incl. identical 1.0) -> same block
            union(a, b)

# collect members of each block
blocks = {}
for c in missing_cols:
    blocks.setdefault(find(c), []).append(c)

# --- 3. collapse each block into one reportable flag ---
def block_name(cols):
    nums = [float(re.match(r"([\d.]+)", c).group(1)) for c in cols]
    lo, hi = min(nums), max(nums)
    return f"skipped_block_{lo:g}" if lo == hi else f"skipped_block_{lo:g}_{hi:g}"

created = []
for members in blocks.values():
    name = block_name(members)
    df[name] = df[members].max(axis=1)     # 1 if missing on any member
    created.append((name, members))

# --- 4. drop the originals, report what happened ---
df = df.drop(columns=missing_cols)

for name, members in sorted(created):
    print(f"{name:28s}  <- {len(members)} cols: {members}")

In [ ]:
def nps_reverse(score):
    if score in [1, 2]:
        return "promotors"
    elif score in [3, 4]:
        return "passives"
    else:
        return "detractors"

df["nps_category"] = df["2001"].apply(nps_reverse)


In [ ]:
cols_to_drop = ["2001"]
df = df.drop(columns=cols_to_drop)

In [ ]:
# ─────────── STEP 1: Identify target and excluded columns ───────────
target_col = 'nps_category'
df_proc = df.copy()

columns_to_process = [c for c in df_proc.columns if c != target_col]

for col in columns_to_process:
    df_proc[col] = pd.to_numeric(df_proc[col], errors='coerce')

X = df_proc.drop(columns=[target_col])
Y = df_proc[target_col]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)
# Binary columns
binary_cols = [col for col in binary if col in X_train.columns]

binary_imputer = SimpleImputer(strategy="most_frequent")
X_train[binary_cols] = binary_imputer.fit_transform(X_train[binary_cols])
X_test[binary_cols] = binary_imputer.transform(X_test[binary_cols])

# All remaining columns
numeric_cols = [col for col in X_train.columns if col not in binary_cols]

median_imputer = SimpleImputer(strategy="median")
X_train[numeric_cols] = median_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = median_imputer.transform(X_test[numeric_cols])

In [ ]:
# ───────────────────────────────────────────────────────────────
# STEP 1: Import libraries

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import shap
import numpy as np

# STEP 2: Define the model

rf = RandomForestClassifier(random_state=42)

# STEP 3: Define the hyperparameter grid
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# STEP 4: Grid search with cross-validation
rf_cv = GridSearchCV(
    rf,
    param_grid=params,
    cv=10,
    scoring='accuracy',   # you can change to 'f1_macro' for imbalanced classes
    n_jobs=-1
)

rf_cv.fit(X_train, Y_train)

print("✅ Best parameters:", rf_cv.best_params_)
print("✅ Best cross-val Accuracy:", rf_cv.best_score_)

# ───────────────────────────────────────────────────────────────
# STEP 5: Use the best model
# ───────────────────────────────────────────────────────────────
best_rf = rf_cv.best_estimator_


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
import matplotlib.pyplot as plt

# Predictions on the test set
Y_pred = best_rf.predict(X_test)

# Accuracy
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# F1-score (macro)
f1 = f1_score(Y_test, Y_pred, average='macro')
print(f"Test F1-score (macro): {f1:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

# Confusion matrix
cm = confusion_matrix(Y_test, Y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.base import clone

# --- detect flag columns (handles both naming states) ---
missing_flags = [c for c in X_train.columns
                 if c.startswith("skipped_block_") or c.endswith("_missing_all")]
print(f"{len(missing_flags)} flag columns will be dropped:", missing_flags[:8])
assert len(missing_flags) > 0, "No flag columns found — check the prefix/suffix names!"

# --- same hyperparameters as best_rf, retrained WITHOUT the flags ---
X_train_wo = X_train.drop(columns=missing_flags)
X_test_wo  = X_test.drop(columns=missing_flags)

rf_wo = clone(best_rf).fit(X_train_wo, Y_train)
Y_pred_wo = rf_wo.predict(X_test_wo)

# --- side-by-side comparison on the SAME test set ---
f1_with = f1_score(Y_test, best_rf.predict(X_test), average="macro")
f1_wo   = f1_score(Y_test, Y_pred_wo, average="macro")

print(f"\nmacro-F1 WITH flags:    {f1_with:.4f}")
print(f"macro-F1 WITHOUT flags: {f1_wo:.4f}")
print(f"difference (with − without): {f1_with - f1_wo:+.4f}")

print("\nClassification report WITHOUT flags:")
print(classification_report(Y_test, Y_pred_wo))

In [ ]:
print(best_rf.classes_)


In [ ]:
explainer = shap.TreeExplainer(best_rf)   # ← this line defines 'explainer'

In [ ]:
shap_values = explainer.shap_values(X_train)


In [ ]:
class_idx = 1  # <- change to the class you want to visualize
sv_class = shap_values[:, :, class_idx]   # (n_samples, n_features)


In [ ]:
import shap
shap.summary_plot(sv_class, X_train, plot_type="dot", max_display=30)


In [ ]:
# Mean absolute SHAP over samples AND classes
global_mean_abs = np.abs(shap_values).mean(axis=(0, 2))  # shape: (n_features,)

global_top10 = (pd.DataFrame({
    "Feature": X_train.columns,
    "Mean |SHAP| (global)": global_mean_abs
})
.sort_values("Mean |SHAP| (global)", ascending=False)
.head(15))


In [ ]:
# re-read question file with IDs as string so 312.10 stays 312.10 (not float 312.1)
df_pitanja = pd.read_excel("question.xlsx", dtype={'QuestionID': str})

global_top10['Feature'] = global_top10['Feature'].astype(str)
df_pitanja['QuestionID'] = df_pitanja['QuestionID'].astype(str)

global_top10 = global_top10.merge(
    df_pitanja[['QuestionID', 'Text']],
    left_on='Feature', right_on='QuestionID', how='left'
)

In [ ]:
global_top10.to_excel("xy.xlsx", index=False)


In [ ]:
global_top10

In [ ]:
# from google.colab import files
# files.download("xy.xlsx")
